In [46]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score
)

### 데이터 불러오기

In [47]:
# 파일 불러오기 
target = "Risk_Label"

# Load datasets
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]


print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)


train shape: (2486, 10) (2486,)
valid shape: (1438, 10) (1438,)
test shape: (822, 10) (822,)


## Grid Search SVM 적합

In [48]:
from sklearn.model_selection import ParameterGrid
from sklearn.base import clone

# SVM pipeline
# 주의: data_train_adasyn / data_valid / data_test가 이미 scaling 완료된 데이터라고 가정
svm_pipeline = Pipeline(
    steps=[("svm", SVC())]
)

# C=0.01이 기존 탐색 범위의 하한이었으므로 더 작은 C까지 확장
C_grid = [0.001, 0.003, 0.01, 0.03, 0.1, 1, 10, 50, 100]
C_grid_poly = [0.001, 0.003, 0.01, 0.03, 0.1, 1, 10, 50]

# Parameter search space
param_grid = [
    {
        "svm__kernel": ["linear"],
        "svm__C": C_grid,
        "svm__class_weight": [None],
    },
    {
        "svm__kernel": ["rbf"],
        "svm__C": C_grid,
        "svm__gamma": ["scale", 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1],
        "svm__class_weight": [None],
    },
    {
        "svm__kernel": ["poly"],
        "svm__C": C_grid_poly,
        "svm__gamma": ["scale", 0.001, 0.003, 0.01, 0.03, 0.1],
        "svm__degree": [2, 3],
        "svm__coef0": [0, 1],
        "svm__class_weight": [None],
    }
]

best_valid_gm = -1.0
best_valid_recall = -1.0
best_params = None
search_results = []

# Train set으로 학습하고 validation set의 G-Mean 기준으로 best model 선택
# G-Mean이 같으면 high-risk recall이 더 높은 모델을 선택
for params in ParameterGrid(param_grid):
    model = clone(svm_pipeline).set_params(**params)
    model.fit(X_train, y_train)

    valid_pred = model.predict(X_valid)

    cm = confusion_matrix(y_valid, valid_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    g_mean = np.sqrt(sensitivity * specificity)

    acc = accuracy_score(y_valid, valid_pred)
    precision = precision_score(y_valid, valid_pred, zero_division=0)
    recall = recall_score(y_valid, valid_pred, zero_division=0)
    f1 = f1_score(y_valid, valid_pred, zero_division=0)

    search_results.append({
        **params,
        "valid_accuracy": acc,
        "valid_precision": precision,
        "valid_recall": recall,
        "valid_f1": f1,
        "valid_g_mean": g_mean,
    })

    if (g_mean > best_valid_gm) or (
        np.isclose(g_mean, best_valid_gm) and recall > best_valid_recall
    ):
        best_valid_gm = g_mean
        best_valid_recall = recall
        best_params = params

# Best SVM model 재학습
best_svm = clone(svm_pipeline).set_params(**best_params)
best_svm.fit(X_train, y_train)

# Validation 최종 성능
valid_pred = best_svm.predict(X_valid)

cm_valid = confusion_matrix(y_valid, valid_pred, labels=[0, 1])
tn, fp, fn, tp = cm_valid.ravel()

valid_acc = accuracy_score(y_valid, valid_pred)
valid_precision = precision_score(y_valid, valid_pred, zero_division=0)
valid_recall = recall_score(y_valid, valid_pred, zero_division=0)
valid_f1 = f1_score(y_valid, valid_pred, zero_division=0)

valid_sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
valid_specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
valid_g_mean = np.sqrt(valid_sensitivity * valid_specificity)

search_results_df = pd.DataFrame(search_results).sort_values(
    ["valid_g_mean", "valid_recall"],
    ascending=False
)

print("\n=== Best SVM Model by Validation G-Mean ===")
print("Best Params:", best_params)
print(f"Best Valid G-Mean: {best_valid_gm:.4f}")

print("\n[Validation] Metrics")
print(f"Accuracy  : {valid_acc:.4f}")
print(f"Precision : {valid_precision:.4f}")
print(f"Recall    : {valid_recall:.4f}")
print(f"F1-Score  : {valid_f1:.4f}")
print(f"G-Mean    : {valid_g_mean:.4f}")

print("\n[Validation] Confusion Matrix")
print(cm_valid)

display(search_results_df.head(10))


=== Best SVM Model by Validation G-Mean ===
Best Params: {'svm__C': 0.01, 'svm__class_weight': None, 'svm__coef0': 0, 'svm__degree': 3, 'svm__gamma': 'scale', 'svm__kernel': 'poly'}
Best Valid G-Mean: 0.5876

[Validation] Metrics
Accuracy  : 0.8561
Precision : 0.3653
Recall    : 0.3765
F1-Score  : 0.3708
G-Mean    : 0.5876

[Validation] Confusion Matrix
[[1170  106]
 [ 101   61]]


,svm__C,svm__class_weight,svm__kernel,valid_accuracy,valid_precision,valid_recall,valid_f1,valid_g_mean,svm__gamma,svm__coef0,svm__degree
135,0.01,None,poly,0.856050,0.365269,0.376543,0.370821,0.587591,scale,0.0,3.0
243,10.00,None,poly,0.821280,0.283105,0.382716,0.325459,0.579333,scale,1.0,3.0
147,0.01,None,poly,0.858136,0.368750,0.364198,0.366460,0.579111,scale,1.0,3.0
177,0.10,None,poly,0.854659,0.357576,0.364198,0.360856,0.577878,scale,0.0,2.0
254,50.00,None,poly,0.854659,0.357576,0.364198,0.360856,0.577878,0.1,0.0,2.0
231,10.00,None,poly,0.812239,0.267241,0.382716,0.314721,0.575958,scale,0.0,3.0
248,10.00,None,poly,0.858136,0.367089,0.358025,0.362500,0.574427,0.1,1.0,3.0
189,0.10,None,poly,0.856745,0.362500,0.358025,0.360248,0.573938,scale,1.0,2.0
271,50.00,None,poly,0.855355,0.358025,0.358025,0.358025,0.573449,0.03,1.0,3.0
159,0.03,None,poly,0.858136,0.365385,0.351852,0.358491,0.569696,scale,0.0,3.0


### Test 성능 결과 

In [49]:
# Test prediction/evaluation using tuned model
test_pred = best_svm.predict(X_test)

# Test 지표 계산
cm_test = confusion_matrix(y_test, test_pred, labels=[0, 1])
tn, fp, fn, tp = cm_test.ravel()

test_acc = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
test_g_mean = np.sqrt(sensitivity * specificity)

print("[Test] Metrics")
print(f"Accuracy  : {test_acc:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1-Score  : {test_f1:.4f}")
print(f"G-Mean    : {test_g_mean:.4f}")
print(f"\n[Test] Confusion Matrix:\n", cm_test)


[Test] Metrics
Accuracy  : 0.8345
Precision : 0.2857
Recall    : 0.3297
F1-Score  : 0.3061
G-Mean    : 0.5439

[Test] Confusion Matrix:
 [[656  75]
 [ 61  30]]
